In [3]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import os, io, pickle
from langchain.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    SpacyTextSplitter
)
from langchain.embeddings import HuggingFaceEmbeddings
import tiktoken
import statistics
import io
import os
import fitz  # PyMuPDF
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
#from langchain_community.vectorstores import FAISS
#from langchain_community.embeddings import OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
from google.oauth2.credentials import Credentials
import pandas as pd

In [31]:
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

def authenticate_google():
    creds = None
    if os.path.exists("token"):
        with open("token", "rb") as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials3.json", SCOPES)
            creds = flow.run_local_server(port=0)
        with open("token", "wb") as token:
            pickle.dump(creds, token)
    
    return build("drive", "v3", credentials=creds)

def download_pdf(file_id, output_path):
    service = authenticate_google()
    request = service.files().get_media(fileId=file_id)
    fh = io.FileIO(output_path, "wb")
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while done is False:
        status, done = downloader.next_chunk()
        print(f"Téléchargement : {int(status.progress() * 100)}%")
    print(f"Fichier téléchargé : {output_path}")


file_id = "1AHE1lXi_kyrtRw31qEGJ7OYE9EO8kfGe"
download_pdf(file_id, "Histoire_CM1.pdf")

Téléchargement : 100%
Fichier téléchargé : Histoire_CM1.pdf


In [32]:
# Fonction pour estimer le nombre de tokens d’un texte
def count_tokens(text, model="gpt-3.5-turbo"):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [33]:
# Charger ton PDF
loader = PyPDFLoader("Histoire_CM1.pdf",
                     mode = "page", # Extract the PDF by page. Each page is extracted as a langchain Document object
                     # mode = "single" # PyPDFLoader will split the PDF as a single text flow
                     )
docs = loader.load()

In [34]:
docs

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-07-23T15:58:49+02:00', 'title': 'Leçons d’histoire  CM1', 'author': 'Saleur', 'moddate': '2018-07-23T15:58:49+02:00', 'source': 'Histoire_CM1.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  \n \n \n1)  Qu’est-ce que l’Histoire ? \n \n*L’Histoire est l’étude de notre passé  pour mieux \ncomprendre notre vie aujourd’hui. \n \n *Pour découvrir notre passé, les historiens font des fouilles \narchéologiques, étudient des objets, des documents, des récits…  \n*Ils représentent le temps par une ligne graduée  : c’est la frise \nchronologique.  \n \n*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient \nl’Histoire.  \n*L’Histoire de France est divisée en 5 périodes : \nl’Antiquité – le Moyen Âge – les Temps Modernes – le XIX ème \nsiècle – le XXème siècle. \n \n \n \n \n2) Des traces du passé : les

In [35]:
print("Nombre de pages :", len(docs))
print("Métadonnées :", docs[0].metadata)

Nombre de pages : 12
Métadonnées : {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2018-07-23T15:58:49+02:00', 'title': 'Leçons d’histoire  CM1', 'author': 'Saleur', 'moddate': '2018-07-23T15:58:49+02:00', 'source': 'Histoire_CM1.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}


In [ ]:
# Si option mode = "single" 
#full_text = docs[0].page_content
#full_text

In [36]:
# Si mode = "page", pour concaténer tout le contenu du PDF en un seul texte brut
full_text = "\n".join([page.page_content for page in docs])
full_text

'LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  \n \n \n1)  Qu’est-ce que l’Histoire ? \n \n*L’Histoire est l’étude de notre passé  pour mieux \ncomprendre notre vie aujourd’hui. \n \n *Pour découvrir notre passé, les historiens font des fouilles \narchéologiques, étudient des objets, des documents, des récits…  \n*Ils représentent le temps par une ligne graduée  : c’est la frise \nchronologique.  \n \n*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient \nl’Histoire.  \n*L’Histoire de France est divisée en 5 périodes : \nl’Antiquité – le Moyen Âge – les Temps Modernes – le XIX ème \nsiècle – le XXème siècle. \n \n \n \n \n2) Des traces du passé : les grottes ornées \n \n*En 1940, 4 enfants découvrent une grotte recouverte \nde peintures  : des taureaux, des cerfs, des chevaux…  : \nla grotte de Lascaux.  En datant les objets trouvés dans la grotte on \nsait qu’elle a été peinte il y a environ 17000 ans.  \n \n*En 1994, Jean -Marie Chauvet découvre une autre grotte pei

In [37]:
# Définir les splitters à tester
splitters = {
    "RecursiveCharacterTextSplitter": RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50),
    "CharacterTextSplitter": CharacterTextSplitter(separator="\n \n", chunk_size=500, chunk_overlap=50),
    "SpacyTextSplitter": SpacyTextSplitter(pipeline="fr_core_news_sm", chunk_size=500)
}


In [38]:
# Tester les splitters
for name, splitter in splitters.items():
    chunks = splitter.split_text(full_text)
    token_counts = [count_tokens(chunk) for chunk in chunks]

    print(f"\n🧩 {name}")
    print(f" - Nombre de chunks : {len(chunks)}")
    print(f" - Tokens/chunk (moy) : {round(statistics.mean(token_counts))}")
    print(f" - Tokens max : {max(token_counts)}")
    print(f" - Aperçu du 1er chunk :\n{chunks[0]}\n")
    #print(f" - Aperçu du 2e chunk :\n{chunks[1]}\n")
    #print(f" - Aperçu du 3e chunk :\n{chunks[2]}\n")



🧩 RecursiveCharacterTextSplitter
 - Nombre de chunks : 27
 - Tokens/chunk (moy) : 144
 - Tokens max : 161
 - Aperçu du 1er chunk :
LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  
 
 
1)  Qu’est-ce que l’Histoire ? 
 
*L’Histoire est l’étude de notre passé  pour mieux 
comprendre notre vie aujourd’hui. 
 
 *Pour découvrir notre passé, les historiens font des fouilles 
archéologiques, étudient des objets, des documents, des récits…  
*Ils représentent le temps par une ligne graduée  : c’est la frise 
chronologique.  
 
*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient 
l’Histoire.


🧩 CharacterTextSplitter
 - Nombre de chunks : 30
 - Tokens/chunk (moy) : 130
 - Tokens max : 162
 - Aperçu du 1er chunk :
LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  
 
 
1)  Qu’est-ce que l’Histoire ? 
 
*L’Histoire est l’étude de notre passé  pour mieux 
comprendre notre vie aujourd’hui. 
 
 *Pour découvrir notre passé, les historiens font des fouilles 
archéologiques, étudient des ob

Pour pouvoir dérouler la suite, on va retenir dans un premier temps la stratégie de chunking basée sur le RecursiveCharacterSplitter sans tokenization.
[Justification à ajouter]


In [39]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Quelle différence entre split_text et split_documents ?
# split_text renvoie une liste de strings correspondant à la liste des morceaux de texte
# split_documents renvoie une liste de "Langchain Documents" (il semble splitter les chunks par page)
# Le nombre de chunks produits est différent : 27 avec split_text, 31 avec split_documents
chunks = splitter.split_text(full_text) 
#chunks = splitter.split_documents(docs)

In [40]:
print(type(chunks))
print(len(chunks))

<class 'list'>
27


In [41]:
chunks

['LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  \n \n \n1)  Qu’est-ce que l’Histoire ? \n \n*L’Histoire est l’étude de notre passé  pour mieux \ncomprendre notre vie aujourd’hui. \n \n *Pour découvrir notre passé, les historiens font des fouilles \narchéologiques, étudient des objets, des documents, des récits…  \n*Ils représentent le temps par une ligne graduée  : c’est la frise \nchronologique.  \n \n*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient \nl’Histoire.',
 'l’Histoire.  \n*L’Histoire de France est divisée en 5 périodes : \nl’Antiquité – le Moyen Âge – les Temps Modernes – le XIX ème \nsiècle – le XXème siècle. \n \n \n \n \n2) Des traces du passé : les grottes ornées \n \n*En 1940, 4 enfants découvrent une grotte recouverte \nde peintures  : des taureaux, des cerfs, des chevaux…  : \nla grotte de Lascaux.  En datant les objets trouvés dans la grotte on \nsait qu’elle a été peinte il y a environ 17000 ans.',
 '*En 1994, Jean -Marie Chauvet découvre une a

In [42]:
print("\n--- Statistiques sur les chunks du document ---")
print(f" - Nombre de chunks : {len(chunks)}")
print(f" - Tokens/chunk (moy) : {round(statistics.mean(token_counts))}")
print(f" - Tokens max : {max(token_counts)}")
print(f" - Aperçu du 1er chunk :\n{chunks[0]}\n")
print(f" - Aperçu du 2e chunk :\n{chunks[1]}\n")
print(f" - Aperçu du 3e chunk :\n{chunks[2]}\n")


--- Statistiques sur les chunks du document ---
 - Nombre de chunks : 27
 - Tokens/chunk (moy) : 129
 - Tokens max : 160
 - Aperçu du 1er chunk :
LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  
 
 
1)  Qu’est-ce que l’Histoire ? 
 
*L’Histoire est l’étude de notre passé  pour mieux 
comprendre notre vie aujourd’hui. 
 
 *Pour découvrir notre passé, les historiens font des fouilles 
archéologiques, étudient des objets, des documents, des récits…  
*Ils représentent le temps par une ligne graduée  : c’est la frise 
chronologique.  
 
*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient 
l’Histoire.

 - Aperçu du 2e chunk :
l’Histoire.  
*L’Histoire de France est divisée en 5 périodes : 
l’Antiquité – le Moyen Âge – les Temps Modernes – le XIX ème 
siècle – le XXème siècle. 
 
 
 
 
2) Des traces du passé : les grottes ornées 
 
*En 1940, 4 enfants découvrent une grotte recouverte 
de peintures  : des taureaux, des cerfs, des chevaux…  : 
la grotte de Lascaux.  En datant

In [43]:
# Nombre de caractères par chunk
print([len(chunk) for chunk in chunks])

# On vérifie que le nombre de caractères est inférieur au chunk_size spécifié (500) 

[474, 443, 442, 481, 486, 472, 443, 489, 444, 491, 445, 456, 453, 445, 451, 450, 498, 490, 494, 485, 474, 449, 461, 448, 435, 463, 208]


In [15]:
from dotenv import load_dotenv
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

KeyError: 'OPENAI_API_KEY'

In [ ]:
"""# problème de package à appeler (peut-être à cause de la version de Python ; j'utilise la 3.13.5)
import sys
!{sys.executable} -m pip install langchain_openai"""

In [ ]:
"""from langchain_openai import OpenAIEmbeddings"""

In [ ]:
# Créer les embeddings de chunks
# le modèle text-embedding-3-small est le moins cher (0.02$ pour 1 M tokens)

embeddings_model = OpenAIEmbeddings(
    openai_api_key = OPENAI_API_KEY,
    model = "text-embedding-3-small"
)

chunk_embeddings = embeddings_model.embed_documents(chunks)


In [16]:
%pip install sentence-transformers

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ---------- ----------------------------- 2.4/8.9 MB 12.2 MB/s eta 0:00:01
   ---------------------- ----------------- 5.0/8.9 MB 12.1 MB/s eta 0:00:01
   ---------------------------------- ----- 7.6/8.9 MB 11.7 MB/s eta 0:00:01
   -------------------------------------- - 8.7/8.9 MB 11.9 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 8.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   -- ------------------------------------- 2.4/41.3 MB 12.2 MB/s eta 0:00:04
   ---- ----------------------------------- 5.0/41.3 MB 12.1 MB/s eta 0:00:04
   ------- -------------------------------- 7.6/41.3 MB 11.7 MB/s eta 0:00:03
   --------- ------------------------------ 10.2/41.3 MB 11.8 MB/s eta 0:00:03
   ------------ --------------------------- 12.6/41.3 MB 11.8 MB/s eta 0:00:03
   -------------

In [44]:
chunks

['LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  \n \n \n1)  Qu’est-ce que l’Histoire ? \n \n*L’Histoire est l’étude de notre passé  pour mieux \ncomprendre notre vie aujourd’hui. \n \n *Pour découvrir notre passé, les historiens font des fouilles \narchéologiques, étudient des objets, des documents, des récits…  \n*Ils représentent le temps par une ligne graduée  : c’est la frise \nchronologique.  \n \n*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient \nl’Histoire.',
 'l’Histoire.  \n*L’Histoire de France est divisée en 5 périodes : \nl’Antiquité – le Moyen Âge – les Temps Modernes – le XIX ème \nsiècle – le XXème siècle. \n \n \n \n \n2) Des traces du passé : les grottes ornées \n \n*En 1940, 4 enfants découvrent une grotte recouverte \nde peintures  : des taureaux, des cerfs, des chevaux…  : \nla grotte de Lascaux.  En datant les objets trouvés dans la grotte on \nsait qu’elle a été peinte il y a environ 17000 ans.',
 '*En 1994, Jean -Marie Chauvet découvre une a

In [ ]:
#test avec hugging face
from sentence_transformers import SentenceTransformer
import numpy as np
print(chunks[:2])

#  Exemple de texte découpé (chunks)
chunks_to_encode = chunks[:2]

# Charger le modèle d'embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')



# Générer les embeddings
embeddings = model.encode(chunks_to_encode)

# Affichage clair
for i, (chunk, emb) in enumerate(zip(chunks_to_encode, embeddings)):
    print("="*60)
    print(f"Chunk {i+1}:")
    print(chunk)
    print(f"\n Embedding (taille {len(emb)}):")
    print(np.round(emb[:10], 3))  # Affiche les 10 premières valeurs, arrondies pour lisibilité
    print("="*60)


['LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  \n \n \n1)  Qu’est-ce que l’Histoire ? \n \n*L’Histoire est l’étude de notre passé  pour mieux \ncomprendre notre vie aujourd’hui. \n \n *Pour découvrir notre passé, les historiens font des fouilles \narchéologiques, étudient des objets, des documents, des récits…  \n*Ils représentent le temps par une ligne graduée  : c’est la frise \nchronologique.  \n \n*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient \nl’Histoire.', 'l’Histoire.  \n*L’Histoire de France est divisée en 5 périodes : \nl’Antiquité – le Moyen Âge – les Temps Modernes – le XIX ème \nsiècle – le XXème siècle. \n \n \n \n \n2) Des traces du passé : les grottes ornées \n \n*En 1940, 4 enfants découvrent une grotte recouverte \nde peintures  : des taureaux, des cerfs, des chevaux…  : \nla grotte de Lascaux.  En datant les objets trouvés dans la grotte on \nsait qu’elle a été peinte il y a environ 17000 ans.']
🧩 Chunk 1:
LLeeççoonnss  dd’’hhiissttooiirree  

In [ ]:
# Charger le modèle d'embeddings ici avec Hugging Face
model = SentenceTransformer('all-MiniLM-L6-v2')

# Générer les embeddings
embeddings = model.encode(chunks)

# Affichage clair
for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
    print("="*60)
    print(f"Chunk {i+1}:")
    print(chunk)
    print(f"\n Embedding (taille {len(emb)}):")
    print(np.round(emb[:10], 3))  # Affiche les 10 premières valeurs, arrondies pour lisibilité
    print("="*60)

Chunk 1:
LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  
 
 
1)  Qu’est-ce que l’Histoire ? 
 
*L’Histoire est l’étude de notre passé  pour mieux 
comprendre notre vie aujourd’hui. 
 
 *Pour découvrir notre passé, les historiens font des fouilles 
archéologiques, étudient des objets, des documents, des récits…  
*Ils représentent le temps par une ligne graduée  : c’est la frise 
chronologique.  
 
*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient 
l’Histoire.

 Embedding (taille 384):
[-0.024  0.15   0.059 -0.052 -0.017  0.151 -0.052  0.037 -0.035  0.003]
Chunk 2:
l’Histoire.  
*L’Histoire de France est divisée en 5 périodes : 
l’Antiquité – le Moyen Âge – les Temps Modernes – le XIX ème 
siècle – le XXème siècle. 
 
 
 
 
2) Des traces du passé : les grottes ornées 
 
*En 1940, 4 enfants découvrent une grotte recouverte 
de peintures  : des taureaux, des cerfs, des chevaux…  : 
la grotte de Lascaux.  En datant les objets trouvés dans la grotte on 
sait qu’elle a ét

In [ ]:
# Construire l’index vectoriel FAISS
# code de Nadège ici :
index = FAISS.from_embeddings(chunk_embeddings, chunks)

In [49]:
%pip install faiss-cpu sentence-transformers

   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ------ --------------------------------- 2.4/14.9 MB 11.2 MB/s eta 0:00:02
   ------------- -------------------------- 5.0/14.9 MB 11.6 MB/s eta 0:00:01
   -------------------- ------------------- 7.6/14.9 MB 11.7 MB/s eta 0:00:01
   -------------------------- ------------- 10.0/14.9 MB 11.7 MB/s eta 0:00:01
   --------------------------------- ------ 12.6/14.9 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  14.7/14.9 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  14.7/14.9 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  14.7/14.9 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 14.9/14.9 MB 8.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import faiss
# Création de l'index FAISS avec Hugging Face
dim = embeddings.shape[1]  # 384 pour MiniLM
index = faiss.IndexFlatL2(dim)  # Index basé sur la distance euclidienne
index.add(np.array(embeddings))  # Ajout des vecteurs

print("Index FAISS créé avec", index.ntotal, "vecteurs.")

#  Requête exemple (simule un utilisateur)
query = " où se situe la Gaule ?"
query_embedding = model.encode([query])

# Recherche des chunks les plus similaires (k=1)
D, I = index.search(query_embedding, k=1)

print("\n Résultat de la requête :", query)
print(f"Top match (indice FAISS): {I[0][0]}, Distance: {D[0][0]:.3f}")
print("Chunk retrouvé :", chunks[I[0][0]])


Index FAISS créé avec 27 vecteurs.

 Résultat de la requête :  où se situe la Gaule ?
Top match (indice FAISS): 3, Distance: 0.825
Chunk retrouvé : surtout des rennes dont il utilise tous les éléments. (La peau pour les 
vêtements ou les cabanes, la graisse pour les lampes, les dents en 
collier…) 
 
*C’est un artisan capable de fabriquer de nombreux outils en 
taillant des pierres ou de l’os.  
 
*Nomade : adj : qui ne vit pas toujours au même endroit.  
 
 
 
 
 
4) Qui sont les Gaulois ? 
 
*On parle des Gaulois à partir de 600 ans avant JC.  
*La Gaule est constituée de nombreux peuples qui se 
font souvent la guerre.


In [ ]:
# Alternative à tester d'après la lecture de la doc de Langchain
# https://api.python.langchain.com/en/latest/vectorstores/langchain_community.vectorstores.faiss.FAISS.html#langchain_community.vectorstores.faiss.FAISS.from_embeddings
chunks_embeddings_pairs = zip(chunks, chunk_embeddings)
index_faiss_from_embeddings = FAISS.from_embeddings(chunks_embeddings_pairs, embeddings_model)

In [ ]:
# A tester aussi, en utilisant FAISS.from_texts au lieu de FAISS.from_embeddings
# https://api.python.langchain.com/en/latest/vectorstores/langchain_community.vectorstores.faiss.FAISS.html#langchain_community.vectorstores.faiss.FAISS.from_texts

index_faiss_from_texts = FAISS.from_texts(chunks, embeddings_model)

In [75]:
#charger le modèle embedding avec Langchain
from langchain.schema import Document
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

documents = [Document(page_content=chunk, metadata={"chunk_id": i}) for i, chunk in enumerate(chunks)]



In [76]:
#Création de l’index FAISS dans LangChain
vectorstore = FAISS.from_documents(documents, embedding_model)

# 4. Recherche
query = "Qui est Napoléon ?"
results = vectorstore.similarity_search(query, k=1)

print("Résultat le plus pertinent :")
print(results[0].page_content)

Résultat le plus pertinent :
autre gouvernement est mis en place.  
 
*Il sera renversé par Napoléon le 18 brumaire.   
 
  *Constitution : texte qui précise comment un pays est dirigé.
21) Le Consulat et l’Empire (1799-1815) 
 
*Le général Bonaparte remporte de nombreuses victoires 
militaires, comme Austerlitz.  
 
*Il  adopte une nouvelle constitution qui lui donne tous les 
pouvoirs.  
 
*Il crée les départements dirigés par un préfet, la banque de France, 
les lycées…
